## MIT News (Artificial Intelligence) Scraper

In [1]:
# MIT News Scraper
try:
    from curl_cffi import requests
except ImportError:
    print("Error: curl_cffi not installed. Please run: pip install curl-cffi")
    import requests

from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import time
import re

# ==========================================
# MIT News Scraper Helpers
# ==========================================

def parse_mit_date(date_str):
    """Parse dates like 'December 22, 2025'"""
    if not date_str: return None
    date_str = date_str.strip()
    # Cleanup periods if any, e.g. Dec. 22
    # date_str = date_str.replace(".", "")
    formats = [
        '%B %d, %Y',       # December 22, 2025
        '%b %d, %Y',       # Dec 22, 2025
        '%Y-%m-%d',
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None

def normalize_date(d):
    if isinstance(d, str):
        for fmt in ['%Y-%m-%d', '%B %d, %Y', '%b %d, %Y']:
             try:
                 return datetime.strptime(d, fmt)
             except:
                 continue
    return d

def get_mit_news_list(start_date, end_date=None):
    """
    Scrapes article list from https://news.mit.edu/topic/artificial-intelligence2
    """
    url = "https://news.mit.edu/topic/artificial-intelligence2"
    
    try:
        start_dt = normalize_date(start_date)
        end_dt = normalize_date(end_date) if end_date else datetime.now()
        if start_dt: start_dt = start_dt.replace(hour=0, minute=0, second=0)
        if end_dt: end_dt = end_dt.replace(hour=23, minute=59, second=59)
        
        print(f"Fetching MIT News List... Range: {start_dt} ~ {end_dt}")
        
        # MIT News is usually open but let's use chrome impersonation for safety
        response = requests.get(url, impersonate="chrome", timeout=30)
        if response.status_code != 200:
             print(f"Failed to access list page: {response.status_code}")
             return []
            
        soup = BeautifulSoup(response.content, 'html.parser')
        articles = []
        
        # Selector based on analysis:
        # Container: article.term-page--news-article--item
        items = soup.find_all('article', class_=lambda c: c and 'term-page--news-article--item' in c)
        
        for item in items:
            try:
                # Title & URL
                # h3.term-page--news-article--item--title a
                title_el = item.find('h3', class_=lambda c: c and 'term-page--news-article--item--title' in c)
                link = item.find('a', class_=lambda c: c and 'term-page--news-article--item--title--link' in c)
                
                if not link and title_el:
                     link = title_el.find('a')
                
                if not link: continue
                
                href = link.get('href')
                full_url = f"https://news.mit.edu{href}" if href.startswith('/') else href
                title = link.get_text(strip=True)
                
                # Date
                # p.term-page--news-article--item--publication-date time
                date_el = item.find('time')
                date_str = ""
                found_date = None
                
                if date_el:
                    # Try datetime attribute first
                    dt_attr = date_el.get('datetime')
                    text_date = date_el.get_text(strip=True)
                    
                    if dt_attr:
                        # Format: 2025-12-22T22:20:00Z
                        try:
                            found_date = datetime.strptime(dt_attr.split('T')[0], '%Y-%m-%d')
                            found_date_str = text_date if text_date else found_date.strftime('%B %d, %Y')
                        except:
                            pass
                            
                    if not found_date:    
                        found_date = parse_mit_date(text_date)
                        found_date_str = text_date
                
                # Filter
                if found_date:
                    if start_dt <= found_date <= end_dt:
                         if not any(a['url'] == full_url for a in articles):
                             articles.append({
                                 'url': full_url,
                                 'date': found_date_str,
                                 'dt': found_date,
                                 'title': title
                             })
            except Exception as e:
                print(f"Error parsing item: {e}")
                continue

        print(f"Found {len(articles)} matching articles in list.")
        return articles
        
    except Exception as e:
        print(f"Error in get_mit_news_list: {e}")
        return []

def scrape_mit_article_detail(url, list_info):
    """
    Scrapes detail of a single MIT article.
    """
    try:
        response = requests.get(url, impersonate="chrome", timeout=30)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Main Content
        # .news-article--content--body
        content_div = soup.find(class_=lambda c: c and 'news-article--content--body' in c)
        
        if not content_div:
             # Fallback
             content_div = soup.find('div', itemprop='articleBody')
        
        if content_div:
            # Clean Noise
            for tag in content_div(['script', 'style', 'button', 'nav', 'footer', 'header', 'form', 'iframe']):
                tag.decompose()
            # Also remove "Press Inquiries" or similar footers often found in MIT news
            # But getting text is usually safe enough
            raw_content = content_div.get_text(separator='\n', strip=True)
        else:
            raw_content = ""
            
        # Title extraction from Page (verification)
        h1 = soup.find('h1')
        page_title = h1.get_text(strip=True) if h1 else list_info['title']
        
        return {
            "raw_content": raw_content,
            "source_url": url,
            "date": list_info['date'],
            "raw_title": page_title,
            "source_name": "MIT News"
        }

    except Exception as e:
        print(f"Error scraping article {url}: {e}")
        return None

def run_mit_scraper(start_date, end_date=None):
    target_articles = get_mit_news_list(start_date, end_date)
    results = []
    
    for article in target_articles:
        print(f"  > Scraping: {article['title']} ({article['date']})")
        details = scrape_mit_article_detail(article['url'], article)
        if details:
             results.append(details)
        time.sleep(1)
        
    print(f"Done. Scraped {len(results)} articles.")
    return results


In [2]:
# Test MIT Scraper
if 'run_mit_scraper' in globals():
    # Example Range
    start_date = '2025-12-20'
    end_date = '2026-01-31'
    
    print(f"Running scraper from {start_date} to {end_date}...")
    res = run_mit_scraper(start_date, end_date)
    
    print("\n[Results]")
    for r in res:
        print(r)


Running scraper from 2025-12-20 to 2026-01-31...
Fetching MIT News List... Range: 2025-12-20 00:00:00 ~ 2026-01-31 23:59:59
Found 1 matching articles in list.
  > Scraping: MIT in the media: 2025 in review (December 22, 2025)
Done. Scraped 1 articles.

[Results]
{'raw_content': "“At MIT, innovation ranges from awe-inspiring technology to down-to-Earth creativity,” noted\nChronicle\n, during a campus visit this year for an episode of the program. In 2025, MIT researchers made headlines across print publications, podcasts, and video platforms for key scientific advances, from breakthroughs in quantum and artificial intelligence to new efforts aimed at improving pediatric health care and cancer diagnosis.\nMIT faculty, researchers, students, alumni and staff helped demystify new technologies, highlighted the practical hands-on learning the Institute is known for, and shared what inspires their research with viewers, readers and listeners around the world. Below is a sampling of news momen

In [3]:
r

{'raw_content': "“At MIT, innovation ranges from awe-inspiring technology to down-to-Earth creativity,” noted\nChronicle\n, during a campus visit this year for an episode of the program. In 2025, MIT researchers made headlines across print publications, podcasts, and video platforms for key scientific advances, from breakthroughs in quantum and artificial intelligence to new efforts aimed at improving pediatric health care and cancer diagnosis.\nMIT faculty, researchers, students, alumni and staff helped demystify new technologies, highlighted the practical hands-on learning the Institute is known for, and shared what inspires their research with viewers, readers and listeners around the world. Below is a sampling of news moments to revisit.\nLet’s take a closer look at MIT: It’s alarming to see such a complex, important institution subject to the whims of today’s politics\nWashington Post\ncolumnist George F. Will reflects on MIT and his view of “the damage that can be done to America